# Hydropathy reference for testing feature 8893

**Kyte–Doolittle hydropathy scale.** Higher values indicate more hydrophobic character; lower values indicate more hydrophilic character on this scale. Values are unnormalized.

Source: [ExPASy ProtScale — Kyte & Doolittle](https://web.expasy.org/protscale/pscale/Hphob.Doolittle.html), citing Kyte J. & Doolittle R.F., *J. Mol. Biol.* **157**, 105–132 (1982).

| Amino acid | One-letter code | Hydropathy |
|---|:---:|---:|
| Alanine | A | 1.8 |
| Arginine | R | -4.5 |
| Asparagine | N | -3.5 |
| Aspartic acid | D | -3.5 |
| Cysteine | C | 2.5 |
| Glutamine | Q | -3.5 |
| Glutamic acid | E | -3.5 |
| Glycine | G | -0.4 |
| Histidine | H | -3.2 |
| Isoleucine | I | 4.5 |
| Leucine | L | 3.8 |
| Lysine | K | -3.9 |
| Methionine | M | 1.9 |
| Phenylalanine | F | 2.8 |
| Proline | P | -1.6 |
| Serine | S | -0.8 |
| Threonine | T | -0.7 |
| Tryptophan | W | -0.9 |
| Tyrosine | Y | -1.3 |
| Valine | V | 4.2 |

**Calculation:** centered sliding-window arithmetic mean, default **19 residues**, with equal weights and no normalization. [ProtScale documentation](https://web.expasy.org/protscale/protscale-doc.html) recommends 19 or 21 residues for highlighting hydrophobic membrane-spanning stretches. An odd window gives each mean an exact central residue; try smaller odd windows to inspect shorter stretches.

The function below returns one row per residue with **1-based positions**, the individual residue value (`hydropathy`), and the local mean (`window_mean`). Positions without a complete centered window are `NaN` (the first/last 9 residues for window 19). For a whole-sequence mean, use `profile['hydropathy'].mean()`; local scores retain the stretches that this average hides. Hydropathy alone does not establish membrane topology.

In [ ]:
import pandas as pd


# Unnormalized Kyte–Doolittle values, as listed by ExPASy ProtScale:
# https://web.expasy.org/protscale/pscale/Hphob.Doolittle.html
KYTE_DOOLITTLE = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2,
}


def calculate_hydropathy(sequence: str, window: int = 19) -> pd.DataFrame:
    """Return per-residue Kyte–Doolittle values and centered window means.

    Accepts a plain one-letter sequence; lowercase and whitespace are
    normalized. Rejects empty sequences and symbols outside the 20-residue
    scale (including unknown/ambiguous residues, gaps and stop markers).
    The window must be a positive odd integer no longer than the sequence.
    Positions are 1-based after whitespace removal. Incomplete windows at
    the termini are NaN; individual residue values remain available there.
    """
    if not isinstance(sequence, str):
        raise TypeError('sequence must be a plain one-letter amino acid string.')
    sequence = ''.join(sequence.split()).upper()
    if not sequence:
        raise ValueError('sequence must contain at least one amino acid.')
    unsupported = sorted(set(sequence) - KYTE_DOOLITTLE.keys())
    if unsupported:
        raise ValueError(f'No Kyte–Doolittle value for: {unsupported}')
    if isinstance(window, bool) or not isinstance(window, int):
        raise TypeError('window must be a positive odd integer.')
    if window < 1 or window % 2 == 0:
        raise ValueError('window must be a positive odd integer.')
    if window > len(sequence):
        raise ValueError('window exceeds sequence length; choose a smaller odd window.')

    profile = pd.DataFrame({
        'position': range(1, len(sequence) + 1),
        'residue': list(sequence),
        'hydropathy': [KYTE_DOOLITTLE[aa] for aa in sequence],
    })
    profile['window_mean'] = profile['hydropathy'].rolling(
        window=window, center=True, min_periods=window
    ).mean()
    return profile

In [ ]:
# Synthetic illustration only: hydrophilic / hydrophobic / hydrophilic.
example_profile = calculate_hydropathy('E' * 19 + 'I' * 19 + 'E' * 19)
# Centers of the three stretches: expected means -3.5, 4.5, -3.5.
example_profile.iloc[[9, 28, 47]]